In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
from pathlib import Path
from tqdm.notebook import tqdm

SEED = 692

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

DATASET_FOLDER = "./datasets/"

data_transforms = transforms.Compose([
    transforms.Resize(227),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_dataset = datasets.CIFAR10(
    root=DATASET_FOLDER,
    train=True,
    transform=data_transforms,
    download=True,
)
test_dataset = datasets.CIFAR10(
    root=DATASET_FOLDER,
    train=False,
    transform=data_transforms,
    download=True,
)

train_val_split = 0.8
train_size = int(train_val_split * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=4, shuffle=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

num_samples = 10
train_samples = [train_dataset[i][0] for i in range(num_samples)]

grid_img = make_grid(train_samples, nrow=5, normalize=True)
plt.figure(figsize=(10, 5))
plt.imshow(grid_img.permute(1, 2, 0))
plt.axis("off")
plt.title("Sample Images from CIFAR-10 Training Set")
plt.show()

val_samples = [val_dataset[i][0] for i in range(num_samples)]
grid_img = make_grid(val_samples, nrow=5, normalize=True)
plt.figure(figsize=(10, 5))
plt.imshow(grid_img.permute(1, 2, 0))
plt.axis("off")
plt.title("Sample Images from CIFAR-10 Validation Set")
plt.show()

test_samples = [test_dataset[i][0] for i in range(num_samples)]
grid_img = make_grid(test_samples, nrow=5, normalize=True)
plt.figure(figsize=(10, 5))
plt.imshow(grid_img.permute(1, 2, 0))
plt.axis("off")
plt.title("Sample Images from CIFAR-10 Test Set")
plt.show()

In [ ]:
from torchsummary import summary


class AlexNet(nn.Module):
    def __init__(
        self,
        input_channels: int = 3,
        num_classes: int = 10
    ):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 96, kernel_size=11, stride=4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes),
            nn.LogSoftmax(dim=1)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.head(x)
        return x


model = AlexNet(input_channels=3, num_classes=10).to(device)
summary(model, input_size=(3, 227, 227))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.utils.tensorboard import SummaryWriter
from typing import Optional, Union


def plot_layers_to_tensorboard(
    net: AlexNet,
    writer: SummaryWriter,
    epoch: int,
):
    layers = [
        ("Conv1", net.features[0]),
        ("Conv2", net.features[3]),
        ("Conv3", net.features[6]),
        ("Conv4", net.features[8]),
        ("Conv5", net.features[10]),
    ]
    for name, layer in layers:
        writer.add_histogram(f"{name}/Weights", layer.weight, epoch)
        writer.add_histogram(f"{name}/Biases", layer.bias, epoch)
        if layer.weight.grad is not None:
            writer.add_histogram(f"{name}/Grad", layer.weight.grad, epoch)


def validate(
    net: AlexNet,
    val_dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device = device,
) -> tuple[float, float, np.ndarray, np.ndarray]:
        net.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
    
        labels_cm = []
        predictions_cm = []
        with torch.no_grad():
            for inputs, labels in val_dataloader:
                inputs, labels = inputs.to(device), labels.to(device)
    
                outputs = net(inputs)
                loss = criterion(outputs, labels)
    
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                labels_cm.extend(labels.cpu().numpy())
                predictions_cm.extend(predicted.cpu().numpy())
    
        avg_val_loss = val_loss / total_val
        val_acc = correct_val / total_val
        cm = confusion_matrix(labels_cm, predictions_cm)
        cm_norm = np.round(confusion_matrix(labels_cm, predictions_cm, normalize="true") * 100, 1)
        return avg_val_loss, val_acc, cm, cm_norm


def train(
    net: AlexNet,
    train_dataloader: DataLoader,
    val_dataloader: DataLoader,
    prefix: Optional[str] = None,
    checkpoints_path: Union[str, Path] = "./checkpoints",
    tensorboard_path: str = "./tensorboard",
    epochs: int = 100,
    lr: float = 1e-1,
    upper_bound: float = 1.0,
    device: torch.device = device,
    layers2tensorboard: bool = True,
    lambda_reg: float = 0.0,
):
    checkpoints_path = Path(checkpoints_path)
    checkpoints_path.mkdir(parents=True, exist_ok=True)

    net.to(device)

    criterion = nn.NLLLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, weight_decay=lambda_reg)

    now = datetime.now()
    suffix = now.strftime("%Y-%m-%d_%H-%M-%S")
    prefix = suffix if prefix is None else f"{prefix}_{suffix}"
    writer = SummaryWriter(log_dir=f"{tensorboard_path}/{prefix}")

    accs = {"train": [], "val": []}
    max_acc = 0.0
    best_model_state = None

    writer.add_graph(net, train_dataloader.dataset[0][0].unsqueeze(0).to(device))

    pb = tqdm(range(epochs), total=epochs, desc="Training Progress", unit="epoch")
    for epoch in pb:
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        net.train()
        for i, (x, y) in enumerate(train_dataloader):
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            outputs = net(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += y.size(0)
            correct_train += (predicted == y).sum().item()

            writer.add_scalar("Loss/Train/Batch", loss.item(), epoch * len(train_dataloader) + i)
            writer.add_scalar("Accuracy/Train/Batch", (predicted == y).float().mean().item(), epoch * len(train_dataloader) + i)

        total_train_loss = train_loss / total_train
        train_acc = correct_train / total_train
        writer.add_scalar("Loss/Train/Epoch", total_train_loss, epoch)
        writer.add_scalar("Accuracy/Train/Epoch", train_acc, epoch)

        accs["train"].append(train_acc)

        if layers2tensorboard:
            plot_layers_to_tensorboard(net, writer, epoch)

        val_loss, val_acc, cm, cm_norm = validate(net, val_dataloader, criterion, device)
        accs["val"].append(val_acc)
        writer.add_scalar("Loss/Val/Epoch", val_loss, epoch)
        writer.add_scalar("Accuracy/Val/Epoch", val_acc, epoch)

        pb.set_postfix({
            "Train Loss": f"{total_train_loss:.4f}",
            "Train Acc": f"{train_acc:.4f}",
            "Val Loss": f"{val_loss:.4f}",
            "Val Acc": f"{val_acc:.4f}",
        })

        if val_acc > max_acc:
            max_acc = val_acc
            best_model_state = net.state_dict()
            torch.save(best_model_state, checkpoints_path / f"best_{prefix}.pth")
        
        if val_acc >= upper_bound:
            print(f"Early stopping at epoch {epoch} with validation accuracy {val_acc:.4f} >= upper bound {upper_bound:.4f}")
            break
        
        writer.add_figure("Confusion Matrix/Val", ConfusionMatrixDisplay(cm).plot().figure_, epoch)
        writer.add_figure("Normalized Confusion Matrix/Val", ConfusionMatrixDisplay(cm_norm).plot().figure_, epoch)

    if best_model_state is not None:
        net.load_state_dict(best_model_state)
    
    return accs, best_model_state


In [ ]:
net = AlexNet(input_channels=3, num_classes=10)
epochs = 10
lr = 1e-1
prefix = f"AlexNet-cifar10-e-{epochs}-lr-{lr}"
accs, best_model_state = train(
    net,
    train_loader,
    val_loader,
    prefix=prefix,
    epochs=epochs,
    lr=lr
)

In [ ]:
plt.plot(accs["train"], label="Train Accuracy")
plt.plot(accs["val"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("AlexNet Training and Validation Accuracy on CIFAR-10")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
def test(
    net: AlexNet,
    test_dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device = device,
) -> tuple[float, float, np.ndarray, np.ndarray]:
    net.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0

    labels_cm = []
    predictions_cm = []
    with torch.no_grad():
        for inputs, labels in tqdm(test_dataloader, desc="Testing", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = net(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()
            labels_cm.extend(labels.cpu().numpy())
            predictions_cm.extend(predicted.cpu().numpy())

    avg_test_loss = test_loss / total_test
    test_acc = correct_test / total_test
    cm = confusion_matrix(labels_cm, predictions_cm)
    cm_norm = np.round(confusion_matrix(labels_cm, predictions_cm, normalize="true") * 100, 1)
    return avg_test_loss, test_acc, cm, cm_norm


avg_test_loss, test_acc, cm, cm_norm = test(
    net,
    test_loader,
    nn.NLLLoss(),
    device
)

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
ConfusionMatrixDisplay(cm).plot(ax=plt.gca())
plt.title("Confusion Matrix - Test Set")
plt.subplot(1, 2, 2)
ConfusionMatrixDisplay(cm_norm).plot(ax=plt.gca())
plt.title("Normalized Confusion Matrix - Test Set")
plt.tight_layout()
plt.show()

In [ ]:
def inference(
    net: AlexNet,
    sample: torch.Tensor,
    device: torch.device = device,
) -> torch.Tensor:
    net.eval()
    sample = sample.unsqueeze(0)  # Add batch dimension
    with torch.no_grad():
        inputs = sample.to(device)
        outputs = net(inputs)
        _, predicted = torch.max(outputs.data, 1)
    return predicted.cpu()

sample_idx = np.random.randint(len(test_dataset))
sample_inputs, sample_labels = test_dataset[sample_idx]
input_image = sample_inputs.permute(1, 2, 0).numpy() * 0.5 + 0.5  # Denormalize for visualization
input_label = sample_labels

predicted_labels = inference(net, sample_inputs, device)

plt.imshow(input_image)
plt.title(f"True Label: {input_label}, Predicted Label: {predicted_labels.item()}")
plt.axis("off")
plt.show()